# 🤖 ADP MCP Integration Gateway — Backend Test Suite

**Model Context Protocol v2.1 · Multi-Agent Routing · SCIM RFC 7643**

This notebook installs the backend, starts the FastAPI server, exposes it via a public tunnel,
and runs live tests against all three agent flows:

| # | Flow | Agent | Test Record |
|---|------|-------|-------------|
| 1 | Onboarding — Green Path | Onboarding Agent | Sarah Chen (ID 101) |
| 2 | Onboarding — Amber Path | Onboarding Agent | Elena Vasquez (ID 201) |
| 3 | Payroll Variance | Payroll Agent | John Doe (ID 789) |
| 4 | Schedule Coverage | Scheduling Agent | Shift S-902 |
| 5 | Batch CSV Upload | Onboarding Agent | Mixed 4-row file |

> **No API keys required.** The deterministic dry-run fallback runs all agents locally.
> Set `GEMINI_API_KEY` in the Config cell to enable live LLM reasoning.

---
## ⚙️ Step 1 — Install Dependencies

In [ ]:
%%capture install_output
!pip install fastapi uvicorn[standard] pyngrok requests pydantic --quiet

In [ ]:
import fastapi, uvicorn, pyngrok, requests, pydantic
print(f"✅ fastapi   {fastapi.__version__}")
print(f"✅ uvicorn   {uvicorn.__version__}")
print(f"✅ pyngrok   {pyngrok.__version__}")
print(f"✅ requests  {requests.__version__}")
print(f"✅ pydantic  {pydantic.__version__}")

---
## 📁 Step 2 — Upload Source Files

Upload the five Python source files from your project:
`agent.py`, `app.py`, `batch_manager.py`, `main.py`, `tools.py`

Run the cell below, click **Choose Files**, and select all five.

In [ ]:
from google.colab import files
import os

print("Select agent.py, app.py, batch_manager.py, main.py, tools.py")
uploaded = files.upload()

required = {"agent.py", "batch_manager.py", "main.py", "tools.py"}
found = set(uploaded.keys())
missing = required - found

if missing:
    print(f"⚠️  Still missing: {missing}")
else:
    print("✅ All required backend files uploaded.")
    for name in sorted(found):
        size = len(uploaded[name])
        print(f"   {name:25s}  {size:,} bytes")

---
## 🔑 Step 3 — Configuration

Leave `GEMINI_API_KEY` blank to use the deterministic dry-run fallback (recommended for first run).
Set `NGROK_AUTHTOKEN` to expose the server publicly — get one free at https://dashboard.ngrok.com

In [ ]:
import os

# ── Optional: Gemini LLM (leave blank for dry-run mode) ──────────────────
GEMINI_API_KEY  = ""   # e.g. "AIza..."

# ── Optional: ngrok public tunnel ────────────────────────────────────────
NGROK_AUTHTOKEN = ""   # e.g. "2abc..."

# Apply to environment
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("🔑 Gemini API key configured — live LLM mode active.")
else:
    print("🔧 No Gemini key — dry-run (deterministic) mode active.")

if NGROK_AUTHTOKEN:
    os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
    print("🌐 ngrok token configured — public URL will be available.")
else:
    print("📡 No ngrok token — tests will use localhost:8000 only.")

---
## 🚀 Step 4 — Start the FastAPI Server

In [ ]:
import threading
import time
import uvicorn

SERVER_PORT = 8000

def run_server():
    uvicorn.run(
        "main:app",
        host="0.0.0.0",
        port=SERVER_PORT,
        log_level="warning",   # suppress per-request noise in notebook output
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give uvicorn a moment to bind
time.sleep(3)

import requests as req
health = req.get(f"http://localhost:{SERVER_PORT}/", timeout=5).json()
print(f"✅ Server status : {health['status']}")
print(f"   Service       : {health['service']}")
print(f"   Architecture  : {health['engine_architecture']}")

---
## 🌐 Step 5 — Public Tunnel (optional)

Skip this cell if you only need local testing.  
The public URL can be pasted into Streamlit Community Cloud as `ADP_BACKEND_URL`.

In [ ]:
PUBLIC_URL = f"http://localhost:{SERVER_PORT}"  # default fallback

ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "")
if ngrok_token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(SERVER_PORT)
    PUBLIC_URL = tunnel.public_url
    print(f"🌐 Public URL: {PUBLIC_URL}")
    print()
    print("   ↳ Paste this into Streamlit Cloud as ADP_BACKEND_URL")
    print("   ↳ Or share directly with stakeholders for a live demo")
else:
    print(f"📡 Running locally at {PUBLIC_URL}")
    print("   (Add an NGROK_AUTHTOKEN above to get a public URL)")

---
## 🧪 Step 6 — Test Helpers

Shared utilities used by all test cells below.

In [ ]:
import json
import requests as req
from IPython.display import display, Markdown

BASE = f"http://localhost:{SERVER_PORT}"

def invoke(action: str, payload: dict) -> dict:
    """POST /invoke and return the parsed response."""
    r = req.post(f"{BASE}/invoke", json={"action": action, "input": payload}, timeout=20)
    r.raise_for_status()
    return r.json()

def pretty(data: dict, title: str = ""):
    """Render a dict as a tidy Markdown + JSON block."""
    if title:
        display(Markdown(f"### {title}"))
    print(json.dumps(data, indent=2))

def status_badge(status: str) -> str:
    icons = {
        "complete":           "✅ PROVISIONED",
        "incomplete":         "⚠️  ACTION BLOCKED",
        "variance_diagnosed": "🔴 VARIANCE DIAGNOSED",
        "clear":              "✅ PAYROLL CLEAR",
        "coverage_gap_identified": "🟡 COVERAGE GAP",
        "optimized":          "✅ FULLY STAFFED",
        "error":              "❌ ERROR",
    }
    return icons.get(status, f"ℹ️  {status.upper()}")

print("✅ Test helpers loaded.")

---
## ✅ Test 1 — Onboarding: Green Path (Sarah Chen · ID 101)

All three mandatory fields present and valid. Expect `status: complete` + SCIM payload.

In [ ]:
result = invoke("hire_employee", {"candidateId": "101"})
agent  = result.get("agent_response", {})
status = agent.get("status", "")

display(Markdown(f"## {status_badge(status)}"))
display(Markdown(f"**Validation summary:** {agent.get('validation_summary')}"))

if result.get("scim_schema"):
    display(Markdown("**SCIM User Schema (RFC 7643):**"))
    print(json.dumps(result["scim_schema"], indent=2))

assert status == "complete", f"Expected 'complete', got '{status}'"
assert result.get("scim_schema") is not None, "SCIM payload missing"
print("\n✅ Assertions passed.")

---
## ⚠️ Test 2 — Onboarding: Amber Path (Elena Vasquez · ID 201)

`startDate` is NULL in the ATS record. Expect `status: incomplete` + remediation email draft.

In [ ]:
result = invoke("hire_employee", {"candidateId": "201"})
agent  = result.get("agent_response", {})
status = agent.get("status", "")

display(Markdown(f"## {status_badge(status)}"))
display(Markdown(f"**Validation summary:** {agent.get('validation_summary')}"))
display(Markdown(f"**Missing fields:** `{agent.get('missing_fields')}`"))

if agent.get("remediation_draft"):
    display(Markdown("**Draft remediation email:**"))
    print("─" * 60)
    print(agent["remediation_draft"])
    print("─" * 60)

assert status == "incomplete", f"Expected 'incomplete', got '{status}'"
assert "startDate" in agent.get("missing_fields", []), "startDate should be flagged"
assert agent.get("remediation_draft"), "Remediation email should be present"
print("\n✅ Assertions passed.")

---
## 🔴 Test 3 — Payroll Variance Diagnosis (John Doe · ID 789)

Employee 789 has multi-jurisdiction withholding elections. Expect a diagnosed variance with contributing factors.

In [ ]:
result = invoke("diagnose_pay_variance", {"employeeId": "789"})
agent  = result.get("agent_response", {})
status = agent.get("status", "")

display(Markdown(f"## {status_badge(status)}"))

if agent.get("detected_delta") is not None:
    display(Markdown(f"**Detected delta:** `${agent['detected_delta']}`"))

if agent.get("contributing_factors"):
    display(Markdown("**Contributing factors:**"))
    for factor in agent["contributing_factors"]:
        print(f"  • {factor}")

if agent.get("remediation_action"):
    display(Markdown(f"**Remediation action:** {agent['remediation_action']}"))

assert status in ("variance_diagnosed", "clear"), f"Unexpected status: '{status}'"
print("\n✅ Assertions passed.")

---
## 🟡 Test 4 — Schedule Coverage Orchestration (Shift S-902)

Shift S-902 is seeded as `open_shift` with no assigned worker. Expect a coverage gap with candidate routing.

In [ ]:
result = invoke("orchestrate_schedule_coverage", {"shiftId": "S-902"})
agent  = result.get("agent_response", {})
status = agent.get("status", "")

display(Markdown(f"## {status_badge(status)}"))
display(Markdown(f"**Open shift:** `{agent.get('open_shift_id')}`"))
display(Markdown(f"**Staffing deficit:** {agent.get('staffing_deficit')}"))
display(Markdown(f"**Resolution:** {agent.get('recommended_resolution')}"))

if agent.get("eligible_candidates"):
    display(Markdown(f"**Eligible candidates dispatched:** `{agent['eligible_candidates']}`"))

assert status in ("coverage_gap_identified", "optimized"), f"Unexpected status: '{status}'"
print("\n✅ Assertions passed.")

---
## 📦 Test 5 — Async Batch CSV Processing

Submits a 4-row CSV with mixed valid/invalid records, polls for progress, and displays final results.
Covers the non-blocking job queue added in the v2.1 refactor.

In [ ]:
import time
import io

# Build a representative mixed-outcome CSV in memory
CSV_DATA = """id,firstName,lastName,email,department,jobTitle,startDate
B001,Sarah,Chen,sarah.chen@example.com,Engineering,Senior Cloud Engineer,2026-07-01
B002,Marcus,Vance,marcus.vance@example.com,Product,Technical PM,2026-07-15
B003,Elena,Vasquez,elena.v@example.com,Operations,Compliance Specialist,
B004,Alex,Kowalski,,Engineering,Security Analyst,2026-08-01
"""
# B003 missing startDate → should be blocked
# B004 missing email     → should be blocked

# ── Submit ─────────────────────────────────────────────────────────────
submit_resp = req.post(
    f"{BASE}/invoke/batch-file",
    files={"file": ("test_batch.csv", CSV_DATA.encode(), "text/csv")},
    timeout=15,
)
submit_resp.raise_for_status()
job_id = submit_resp.json()["job_id"]
display(Markdown(f"**Job submitted:** `{job_id}`"))

# ── Poll for completion ────────────────────────────────────────────────
deadline = time.time() + 60
poll = {}
while time.time() < deadline:
    poll = req.get(f"{BASE}/invoke/batch-status/{job_id}", timeout=10).json()
    pct       = poll.get("progress", {}).get("percent", 0)
    completed = poll.get("progress", {}).get("completed", 0)
    total     = poll.get("progress", {}).get("total", 0)
    print(f"\r   Progress: {completed}/{total} rows ({pct}%)   ", end="", flush=True)
    if poll.get("status") == "complete":
        break
    time.sleep(1)

print()  # newline after progress line

# ── Results ────────────────────────────────────────────────────────────
meta = poll.get("metadata", {})
display(Markdown("### Batch Results"))
display(Markdown(
    f"| Metric | Count |\n"
    f"|--------|-------|\n"
    f"| Total evaluated | {meta.get('total_records_evaluated', 0)} |\n"
    f"| ✅ Fully provisioned | {meta.get('fully_provisioned_count', 0)} |\n"
    f"| ⚠️ Action blocked | {meta.get('action_blocked_count', 0)} |"
))

print("\nProvisioned records:")
for rec in poll.get("provisioned_records", []):
    print(f"  ✅ {rec['id']:6s}  {rec['name']}")

print("\nBlocked records:")
for rec in poll.get("blocked_records", []):
    print(f"  ⚠️  {rec['id']:6s}  {rec['name']}  — missing: {rec['missing_fields']}")

# ── Assertions ─────────────────────────────────────────────────────────
assert meta.get("total_records_evaluated") == 4,     "Expected 4 rows"
assert meta.get("fully_provisioned_count") == 2,     "Expected 2 provisioned"
assert meta.get("action_blocked_count")    == 2,     "Expected 2 blocked"
print("\n✅ Assertions passed.")

---
## 🔍 Test 6 — Capabilities & SCIM Debug Endpoints

In [ ]:
# /capabilities — A2A directory
caps = req.get(f"{BASE}/capabilities", timeout=5).json()
display(Markdown("### /capabilities"))
display(Markdown(f"**Agent:** `{caps['agent']}` · **Version:** `{caps['version']}`"))
display(Markdown("**Registered actions:**"))
for cap in caps["active_capabilities"]:
    keywords = ", ".join(f"`{k}`" for k in cap["intent_keywords"])
    print(f"  • {cap['name']}")
    print(f"    {cap['description']}")
    print(f"    Intent keywords: {keywords}")
    print()

# /debug/scim/:id
display(Markdown("### /debug/scim/101"))
scim = req.get(f"{BASE}/debug/scim/101", timeout=5).json()
print(json.dumps(scim["scim_format"], indent=2))

---
## 📊 Test 7 — Full Results Summary

In [ ]:
from IPython.display import display, Markdown

tests = [
    ("Onboarding — Green Path   (ID 101)", "hire_employee",              {"candidateId": "101"}, "complete"),
    ("Onboarding — Amber Path   (ID 201)", "hire_employee",              {"candidateId": "201"}, "incomplete"),
    ("Payroll Variance          (ID 789)", "diagnose_pay_variance",      {"employeeId":  "789"}, None),
    ("Schedule Coverage         (S-902)",  "orchestrate_schedule_coverage", {"shiftId": "S-902"}, None),
]

rows = []
for label, action, payload, expected_status in tests:
    try:
        res    = invoke(action, payload)
        agent  = res.get("agent_response", {})
        status = agent.get("status", res.get("status", "unknown"))
        passed = (expected_status is None) or (status == expected_status)
        rows.append((label, status, "✅ PASS" if passed else "❌ FAIL"))
    except Exception as e:
        rows.append((label, f"ERROR: {e}", "❌ FAIL"))

header = "| Test | Agent Status | Result |\n|------|-------------|--------|\n"
body   = "".join(f"| {r[0]} | `{r[1]}` | {r[2]} |\n" for r in rows)
display(Markdown("## 📊 Test Summary\n\n" + header + body))

passed = sum(1 for r in rows if "PASS" in r[2])
print(f"\n{passed}/{len(rows)} tests passed.")

---
## 🌐 Step 7 — Streamlit Deployment Note

Once you're happy with the backend results, deploy the Streamlit UI to the web:

1. Push all five `.py` files + a `requirements.txt` to a GitHub repo
2. Go to [share.streamlit.io](https://share.streamlit.io) → **New app** → select your repo
3. Set the **main file** to `app.py`
4. Under **Advanced settings → Secrets**, add:
   ```
   ADP_BACKEND_URL = "<your ngrok or Cloud Run URL>"
   GEMINI_API_KEY  = "<optional>"
   ```
5. Deploy — you'll have a shareable `https://yourapp.streamlit.app` URL in under 2 minutes.

### Minimal `requirements.txt`
```
fastapi
uvicorn[standard]
streamlit
requests
pydantic
```